In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq
from scipy.stats import mannwhitneyu

rng = np.random.default_rng(0)
files = sorted(glob.glob(f"{SLIM_DIR}/*_slim.parquet"))
rows = []
for f in files:
    roi = os.path.basename(f).replace("_slim.parquet", "")
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in GRMS if g in names]
    df = pd.read_parquet(f, columns=["chem", "class", "subclass"] + genes)
    df = df[[is_neuron_class(c) for c in df["class"]]]
    cls_of = df.groupby("subclass")["class"].first()
    for chem in ["ALL", "WMB-10Xv2", "WMB-10Xv3"]:
        d = df if chem == "ALL" else df[df["chem"] == chem]
        if len(d) < 2 * MIN_CELLS:
            continue
        cpm = (np.exp2(d[genes].astype("float64")) - 1)
        sums = cpm.groupby(d["subclass"].to_numpy()).sum()
        n = d.groupby("subclass").size()
        tot = cpm.sum()
        N = len(d)
        keep = n[(n >= MIN_CELLS) & (N - n >= MIN_CELLS)].index
        for g in genes:
            mean_s = sums.loc[keep, g] / n.loc[keep]
            mean_o = (tot[g] - sums.loc[keep, g]) / (N - n.loc[keep])
            for s in keep:
                pv = np.nan
                if chem == "ALL":
                    m = (d["subclass"] == s).to_numpy()
                    a_ = d[g].to_numpy()[m]
                    b_ = d[g].to_numpy()[~m]
                    b_ = rng.choice(b_, size=min(20000, len(b_)), replace=False)
                    pv = mannwhitneyu(a_, b_, alternative="greater", method="asymptotic").pvalue if np.ptp(np.r_[a_, b_]) > 0 else 1.0
                rows.append((roi, chem, g, cls_of[s], s, int(n[s]), mean_s[s], mean_o[s], (mean_s[s] + 1) / (mean_o[s] + 1), pv))
    print(roi, len(df), "neurons", flush=True)
res = pd.DataFrame(rows, columns=["roi", "chem", "gene", "class", "subclass", "n_cells", "mean_cpm", "mean_cpm_other_neurons", "fold", "p_greater"])

def bh(p):
    p = np.asarray(p, dtype=float)
    o = np.argsort(p)
    q = np.empty_like(p)
    q[o] = np.minimum.accumulate((p[o] * len(p) / (np.arange(len(p)) + 1))[::-1])[::-1]
    return np.minimum(q, 1)

res["q_greater"] = np.nan
m = res["chem"] == "ALL"
res.loc[m, "q_greater"] = res[m].groupby(["roi", "gene"])["p_greater"].transform(bh)
res.to_csv(f"{RES_DIR}/master_enrichment.csv.gz", index=False)
print(len(res), "rows ->", f"{RES_DIR}/master_enrichment.csv.gz")
top = res[(res["chem"] == "ALL") & (res["q_greater"] < 0.05)].sort_values("fold", ascending=False).groupby("gene").head(5)
print(top[["gene", "roi", "subclass", "n_cells", "mean_cpm", "fold"]].round(2).to_string(index=False))